# Pashto FastSpeech2: train on the full dataset (Kaggle)

A Kaggle GPU session ends after 12 h, so training runs over several sessions. Each session continues from the
newest checkpoint, saves a checkpoint at the end of **every epoch**, and stops by itself (with a checkpoint)
before `HOURS` is up, so the session's output is saved.

**Before the first run**
1. Run `colab_align.ipynb` and upload its `pashto_aligned.zip` as a Kaggle dataset (*Create -> New Dataset*).
2. Here: *Add Input -> Your Work -> Datasets ->* that dataset.
3. Right panel: Accelerator **GPU T4 x2**, Internet **on**.

**Smoke test (~15-40 min of GPU quota):** with `SMOKE_TEST = True`, run all cells in this interactive session
and read the checklist printed at the end. Then **stop the session** (it counts quota while open).

**Real run:** set `SMOKE_TEST = False`, then *Save Version -> Save & Run All (Commit)*. Only committed runs
keep `/kaggle/working`, and only if they finish: keep `HOURS` below the GPU quota you have left this week
(see your Kaggle profile), otherwise Kaggle stops the run first and its output is lost.

**Every later session:** *Add Input -> Your Work -> Notebooks ->* this notebook (its newest version), so the last
output (checkpoints, logs, features) is under `/kaggle/input`, then commit again.

Output (the version's *Output* tab): `output/ckpt/Pashto/<step>.pth.tar` (every 10000th step + the newest 2 epoch
checkpoints, ~350 MB each), `output/log/`, `runs/full/metrics/` (loss curves), `runs/full/stats.json` +
`speakers.json` (needed with a checkpoint for `colab_inference.ipynb`), `preprocessed_data/` (the features).

In [ ]:
SMOKE_TEST = True     # True: ~5 min training test (interactive). False: the real run (commit)
HOURS      = 11.5     # this session's budget; training stops (with a checkpoint) ~15 min before it
STEPS      = 300000   # total steps; in practice the time budget is what stops each session
BATCH_SIZE = 16       # lower to 8 on CUDA out-of-memory (30 s clips)

In [ ]:
import os
REPO_DIR = "/kaggle/temp/pashto-fastspeech2"   # outside /kaggle/working: the code doesn't need saving
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/msds25033/pashto-fastspeech2.git {REPO_DIR}
%cd {REPO_DIR}
!git pull -q && git log -1 --format="%h %s"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!find /kaggle/input -maxdepth 4 | head -20

In [ ]:
!python kaggle_train.py --hours {HOURS} --steps {STEPS} --batch-size {BATCH_SIZE} {'--smoke' if SMOKE_TEST else ''}